In [ ]:
ENSEMBLE_MEMBERS = 600
MODEL = "AIM/CGE V2.2"
SCENARIO = "ENGAGE-Feasibility-Maximum-Effort/Institutional_NZCO2"
FILE = "101_netzero_co2_extended.csv"
N_PROCESSES = 2

# 301: Run MAGICC
In this notebook, we run MAGICC for a single scenario for a given number of ensemble
members for MAGICC7.

This notebook always computes all three quantities needed downstream for a given
scenario, in one execution:

1. `ANTHROPOGENIC` (native `rf_total_runmodus`): all forcers except natural/solar/
   volcanic.
2. `CO2`-only, via QEXTRA.
3. Kyoto-gases-only (`CO2+CH4+N2O+F-Gases`, no Montreal Protocol halogens), via QEXTRA
   - this is what downstream analysis treats as the `"GHG"` flag.

For (2) and (3) we don't use MAGICC's native `rf_total_runmodus` for that flag: native
`CO2` mode would derive CO2's own concentration/forcing under an artificial
all-other-forcers-zeroed background (a different, less relevant question than "CO2's
contribution as it actually plays out inside the real scenario"), and MAGICC's built-in
`"GHG"` runmodus bundles Kyoto gases together with Montreal Protocol halogens, which we
don't want for this project's Kyoto-gases-based net-zero-GHG definition.

Instead, we run once in `ANTHROPOGENIC` mode requesting the individual component
`Effective Radiative Forcing` outputs (CO2, CH4, N2O, F-Gases) alongside GSAT, then feed
two different sums of those harvested component ERFs (CO2 alone; all four) back through
two further MAGICC passes via `rf_total_runmodus = "QEXTRA"` (`file_extra_rf` /
`rf_extra_read`). Crucially, this single `ANTHROPOGENIC` pass's own
`Surface Air Temperature Change` output **is** the `ANTHROPOGENIC` result itself - no
separate native `ANTHROPOGENIC`-only run is needed, since GSAT there is identical
regardless of which extra ERF components we also ask MAGICC to report. So this notebook
runs exactly 3 MAGICC passes per scenario (1 harvest + 2 QEXTRA), not 5 as an earlier,
one-row-per-flag version of this pipeline did (1 dedicated `ANTHROPOGENIC` run + a
separate harvest pass duplicated inside each of the `CO2` and `GHG` rows).

The QEXTRA approach has been verified equivalent to the corresponding native runmodus
for the `Surface Air Temperature Change` output specifically - see
`notes/magicc-kyoto-flag-qextra-equivalence.md` for the full reasoning (in short: GSAT
is driven only by Effective Radiative Forcing, which sums identically regardless of
which runmodus branch computes it; MAGICC's per-species Surface Forcing weighting,
which *would* differ under QEXTRA, is never read back into the temperature calculation).

Non-CO2-GHG contribution is meant to be computed downstream as a **subtraction**
(`GHG` QEXTRA result minus `CO2` QEXTRA result), not via native-mode differencing -
see `notes/magicc-carbon-cycle-feedback-mechanics.md` for why: both terms then come
from the *same* underlying real-scenario forcing (harvested from the one
`ANTHROPOGENIC` pass, exact by construction), so the only residual is the QEXTRA
temperature-response stage's own state-dependent nonlinearity, evaluated near the real
scenario's actual warming level in both terms - as opposed to a native `GHG`/`CO2`
pair, where the forcing itself is recomputed under two different, less-realistic
all-or-nothing backgrounds, or a "direct construction" (non-CO2-only QEXTRA) run, which
evaluates that nonlinearity at an artificially cold state the real world never occupies.

Output: three CSVs per scenario, `{model}_{scenario}_{ANTHROPOGENIC,CO2,GHG}.csv`,
written to `individual_runs_raw/` - **raw**, pre-calibration GSAT, not the final
assessment-calibrated output downstream analysis consumes. Calibration
(`CMIP7ScenarioMIPPostProcessor`, matching each ensemble's median to the AR6/AR7
assessed historical warming) is applied once, separately, by
`304_post_process_gsat.ipynb`, after the full batch - cluster runs plus any local
backfill via `303_run_magicc-missing.ipynb` - is complete for every scenario. It's not
done here per-scenario/per-batch because the calibration shift is a function of exactly
which ensemble members are present in the call that computes it (see `notes/` for the
full trace) - doing it here would bake in a spurious, systematic offset between rows
from a partial/backfilled batch and rows from the main batch, once they're combined.

This notebook is designed to be run standalone (for one scenario at a time, using the
default parameters above) or invoked repeatedly via papermill by
`302_running_helpers.py`, which supplies these parameters per batch row from
`300_prepare_scenarios.ipynb`'s output (one row per scenario/variant - there is no
`MAGICC_FLAG` column any more, since this notebook always produces all three).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("pint.util").setLevel(logging.ERROR)

import multiprocessing
import os
import platform
from functools import partial
from pathlib import Path

import gcages.cmip7_scenariomip.pre_processing.reaggregation.basic
from gcages.cmip7_scenariomip.harmonisation import (
    create_cmip7_scenariomip_global_harmoniser,
)
from gcages.cmip7_scenariomip.infilling import CMIP7ScenarioMIPInfiller
from gcages.cmip7_scenariomip.pre_processing import CMIP7ScenarioMIPPreProcessor
from gcages.cmip7_scenariomip.scm_running import CMIP7ScenarioMIPSCMRunner
from gcages.index_manipulation import split_sectors

import dotenv
import openscm_units
import openscm_runner
import pandas_indexing as pix
import pandas_openscm
import pint
import seaborn as sns
from pandas_openscm.io import load_timeseries_csv
import pandas as pd

# Papermill output handling
import sys
from datetime import datetime

# Import sanitization function and the QEXTRA forcing-file writer
from utils import sanitize_label, write_magicc_extra_rf_file

# Load environment variables
dotenv.load_dotenv()

# Set up output folder
OUTPUT_FOLDER = Path(os.environ["OUTPUT_FOLDER"])
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

In [3]:
BASE_DIR = Path("../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs")
CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS = BASE_DIR / "history_cmip7_scenariomip.csv"

In [4]:
FILE

'102_netzero_co2_extended.csv'

In [5]:
FILE = OUTPUT_FOLDER / FILE

In [6]:
start = load_timeseries_csv(
    FILE,
    index_columns=["model", "scenario", "region", "variable", "unit"],
    out_columns_type=int,
    out_columns_name="year"
)

In [7]:
scenario_to_run = (
    start
    .loc[
        pd.IndexSlice[MODEL, SCENARIO, :, :, :], :
    ]
)

In [8]:
scenario_to_run

year                                                                                                                    2023  \
model        scenario                                           region variable                unit                            
AIM/CGE V2.2 ENGAGE-Feasibility-Maximum-Effort/Institutional... World  Emissions|BC            Mt BC/yr             7.219245   
                                                                       Emissions|C2F6          kt C2F6/yr           2.431873   
                                                                       Emissions|C3F8          kt C3F8/yr           0.721178   
                                                                       Emissions|C4F10         kt C4F10/yr          0.101586   
                                                                       Emissions|C5F12         kt C5F12/yr          0.061383   
                                                                       Emissions|C6F14         kt C6F14/yr          0.062998   
                                                                       Emissions|C7F16         kt C7F16/yr          0.186453   
                                                                       Emissions|C8F18         kt C8F18/yr          0.136638   
                                                                       Emissions|CCl4          kt CCl4/yr          40.297308   
                                                                       Emissions|CF4           kt CF4/yr           15.117718   
                                                                       Emissions|CFC11         kt CFC11/yr         55.993337   
                                                                       Emissions|CFC113        kt CFC113/yr         2.030656   
                                                                       Emissions|CFC114        kt CFC114/yr         0.873953   
                                                                       Emissions|CFC115        kt CFC115/yr         0.027629   
                                                                       Emissions|CFC12         kt CFC12/yr         20.278901   
                                                                       Emissions|CH2Cl2        kt CH2Cl2/yr      1612.125558   
                                                                       Emissions|CH3Br         kt CH3Br/yr        140.486974   
                                                                       Emissions|CH3CCl3       kt CH3CCl3/yr        1.925368   
                                                                       Emissions|CH3Cl         kt CH3Cl/yr       5376.602241   
                                                                       Emissions|CH4           Mt CH4/yr          363.489901   
                                                                       Emissions|CHCl3         kt CHCl3/yr        400.499380   
                                                                       Emissions|CO            Mt CO/yr           774.053434   
                                                                       Emissions|CO2|Biosphere Mt CO2/yr         3627.550667   
                                                                       Emissions|CO2|Fossil    Mt CO2/yr        38712.217040   
                                                                       Emissions|HCFC141b      kt HCFC141b/yr      57.429166   
                                                                       Emissions|HCFC142b      kt HCFC142b/yr      20.176795   
                                                                       Emissions|HCFC22        kt HCFC22/yr       324.526866   
                                                                       Emissions|HFC125        kt HFC125/yr       124.074300   
                                                                       Emissions|HFC134a       kt HFC134a/yr      291.788600   
                                

In [9]:
MAGICC_EXE_PATH = Path(
    "../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs/magicc-v7.6.0a3/bin"
)
MAGICC_CMIP7_SCENARIOMIP_PROBABILISTIC_CONFIG_FILE = Path(
    "../../gcages/tests/regression/cmip7-scenariomip/cmip7-scenariomip-workflow-inputs/magicc-v7.6.0a3/configs/magicc-ar7-fast-track-drawnset-v0-3-0.json"
)

In [10]:
if platform.system() == "Darwin":
    if platform.processor() == "arm":
        MAGICC_EXE = MAGICC_EXE_PATH / "magicc-darwin-arm64"
        os.environ["DYLD_LIBRARY_PATH"] = "/opt/homebrew/Cellar/gcc/15.2.0_1/lib/gcc/current"

elif platform.system() == "Linux":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc"

elif platform.system() == "Windows":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc.exe"

In [ ]:
CO2_EFFRF_COMPONENTS = (
    "Effective Radiative Forcing|CO2",
)

KYOTO_EFFRF_COMPONENTS = (
    "Effective Radiative Forcing|CO2",
    "Effective Radiative Forcing|CH4",
    "Effective Radiative Forcing|N2O",
    "Effective Radiative Forcing|F-Gases",
)

# The two QEXTRA baskets built from the single ANTHROPOGENIC harvest pass, and the
# component ERFs summed to build each one's driving forcing. Keys double as the
# "flag" label used in output filenames (alongside "ANTHROPOGENIC" itself).
QEXTRA_BASKETS = {
    "CO2": CO2_EFFRF_COMPONENTS,
    "GHG": KYOTO_EFFRF_COMPONENTS,
}

BASKET_DESCRIPTION = {
    "CO2": "CO2-only",
    "GHG": "Kyoto-gases-only (CO2+CH4+N2O+F-Gases, no Montreal halogens)",
}


def build_scm_runner(output_variables):
    """Build a scm runner truncated to ENSEMBLE_MEMBERS. Each call reloads the same
    probabilistic config JSON, so member order/count (and therefore run_id alignment
    across separately-built runners) is deterministic."""
    runner = CMIP7ScenarioMIPSCMRunner.from_cmip7_scenariomip_config(
        n_processes=N_PROCESSES,
        magicc_exe_path=MAGICC_EXE,
        magicc_prob_distribution_path=MAGICC_CMIP7_SCENARIOMIP_PROBABILISTIC_CONFIG_FILE,
        historical_emissions_path=CMIP7_SCENARIOMIP_GLOBAL_HISTORICAL_EMISSIONS,
        output_variables=output_variables,
        batch_size_scenarios=15,
    )
    runner.climate_models_cfgs["MAGICC7"] = runner.climate_models_cfgs["MAGICC7"][:ENSEMBLE_MEMBERS]
    return runner


def sum_effrf_components(erf_results, components):
    """Sum a basket of component ERF series per ensemble member (run_id) and year."""
    summed = (
        erf_results
        .loc[pix.isin(variable=list(components))]
        .groupby(level="run_id")
        .sum(numeric_only=True)
    )
    year_columns = [c for c in summed.columns if isinstance(c, int)]
    return summed[year_columns]

In [ ]:
# Pass 1: single ANTHROPOGENIC-mode run harvesting GSAT (this run's own
# Surface Air Temperature Change output IS the "ANTHROPOGENIC" flag's result - see
# notebook intro) plus the component ERFs needed to build both QEXTRA baskets below.
erf_runner = build_scm_runner(
    output_variables=("Surface Air Temperature Change", *KYOTO_EFFRF_COMPONENTS)
)
erf_runner.climate_models_cfgs["MAGICC7"] = [
    {**c, "rf_total_runmodus": "ANTHROPOGENIC"}
    for c in erf_runner.climate_models_cfgs["MAGICC7"]
]

# The full set of ensemble members we intended to run - gcages sets each config's
# "run_id" key when loading the probabilistic distribution file (from that member's
# own "paraset_id" in the raw JSON, but "run_id" is the key actually present on the
# config dicts we work with here) - a stable, intrinsic identifier per parameter draw,
# not merely list position, so this is safe to use as the "expected" set for tracking
# dropouts below, and for correctly re-selecting configs by run_id even if some are
# missing.
expected_run_ids = sorted(c["run_id"] for c in erf_runner.climate_models_cfgs["MAGICC7"])

# A small, scenario/member-dependent fraction of MAGICC probabilistic ensemble members
# can fail with a native "NaN forcing found" fatal error for reasons that are still
# under investigation (see notes/ and the upstream GitLab issue) - this is not
# something our own code can prevent, so rather than crash the whole scenario, we
# proceed with whichever members survive and track the gap explicitly (see the save
# step below) for a later targeted backfill re-run.
#
# For a small number of scenarios (an unusually extreme emissions trajectory, most
# likely), EVERY member can fail this way rather than just a fraction -
# gcages/openscm-runner then raises (nothing to return at all) instead of giving back
# a degraded result. We still want this tracked for backfill exactly like a partial
# failure, so on that specific failure mode we record all expected_run_ids as missing
# for all three flags before re-raising - the scenario still shows up as an overall
# failed run in 302's own reporting, unchanged from before; this only additionally
# leaves behind the missing_members data the backfill step needs.
sanitized_model = sanitize_label(MODEL)
sanitized_scenario = sanitize_label(SCENARIO)
missing_members_dir = OUTPUT_FOLDER / "missing_members"
missing_members_dir.mkdir(parents=True, exist_ok=True)

try:
    erf_results = erf_runner(scenario_to_run)
except Exception:
    for _flag in ("ANTHROPOGENIC", "CO2", "GHG"):
        pd.DataFrame({
            "model": MODEL,
            "scenario": SCENARIO,
            "flag": _flag,
            "run_id": expected_run_ids,
        }).to_csv(
            missing_members_dir / f"{sanitized_model}_{sanitized_scenario}_{_flag}.csv",
            index=False,
        )
    print(
        f"ERROR: ANTHROPOGENIC harvest pass failed entirely (0/{len(expected_run_ids)} "
        "members) - all members tracked as missing for all three flags; re-raising."
    )
    raise

anthropogenic_gsat = erf_results.loc[pix.isin(variable=["Surface Air Temperature Change"])]

n_harvest_survivors = anthropogenic_gsat.index.get_level_values("run_id").nunique()
if n_harvest_survivors < len(expected_run_ids):
    n_missing = len(expected_run_ids) - n_harvest_survivors
    pct = 100 * n_missing / len(expected_run_ids)
    print(
        f"WARNING: {n_missing}/{len(expected_run_ids)} ({pct:.1f}%) member(s) missing "
        f"from the ANTHROPOGENIC harvest pass - proceeding with {n_harvest_survivors} "
        "survivors; gap will be tracked for backfill."
    )

In [ ]:
qextra_effrf_by_flag = {
    flag: sum_effrf_components(erf_results, components)
    for flag, components in QEXTRA_BASKETS.items()
}
qextra_effrf_by_flag["GHG"]

In [ ]:
sanitized_model = sanitize_label(MODEL)
sanitized_scenario = sanitize_label(SCENARIO)

qextra_gsat_by_flag = {}
for flag, qextra_effrf in qextra_effrf_by_flag.items():
    # Pass 2 (one per basket): write one QEXTRA forcing file per ensemble member,
    # then rerun with rf_total_runmodus="QEXTRA" so each member is driven by exactly
    # its own basket-only forcing series (matching run_id with the harvest pass).
    #
    # The forcing directory includes the flag: both baskets are built from the same
    # harvest and executed within this same notebook run, so a flag-less shared
    # directory would let the CO2 and GHG passes overwrite each other's member
    # forcing files.
    qextra_forcing_dir = (
        OUTPUT_FOLDER / "qextra_forcing" / f"{sanitized_model}_{sanitized_scenario}_{flag}"
    )

    for run_id, row in qextra_effrf.iterrows():
        write_magicc_extra_rf_file(
            qextra_forcing_dir / f"member_{run_id}.IN",
            years=list(row.index),
            values=list(row.values),
            description=(
                f"{BASKET_DESCRIPTION[flag]} ERF: {MODEL} | {SCENARIO}, ensemble member {run_id}"
            ),
        )

    qextra_runner = build_scm_runner(output_variables=("Surface Air Temperature Change",))
    # Key by each config's own "paraset_id" (== run_id) rather than list position, so
    # selecting the subset that survived the harvest pass (qextra_effrf.index) can
    # never misalign a member's forcing data with a different member's MAGICC config -
    # safe even when the harvest pass itself already dropped some members (see above).
    qextra_cfgs_by_id = {c["run_id"]: c for c in qextra_runner.climate_models_cfgs["MAGICC7"]}
    survived_run_ids = [rid for rid in qextra_effrf.index if rid in qextra_cfgs_by_id]

    qextra_runner.climate_models_cfgs["MAGICC7"] = [
        {
            **qextra_cfgs_by_id[run_id],
            "file_extra_rf": str(qextra_forcing_dir / f"member_{run_id}.IN"),
            "rf_extra_read": 1,
            "rf_total_runmodus": "QEXTRA",
        }
        for run_id in survived_run_ids
    ]
    # As with the harvest pass, a small number of scenarios can lose every member of
    # a specific QEXTRA pass rather than just a fraction - gcages/openscm-runner then
    # raises instead of returning a degraded result. Record the full attempted set as
    # missing for this flag and move on to the next flag, rather than crashing the
    # whole scenario over one flag's total failure (the other flags may well still
    # succeed independently).
    try:
        qextra_gsat_by_flag[flag] = qextra_runner(scenario_to_run)
    except Exception:
        pd.DataFrame({
            "model": MODEL,
            "scenario": SCENARIO,
            "flag": flag,
            "run_id": survived_run_ids,
        }).to_csv(
            missing_members_dir / f"{sanitized_model}_{sanitized_scenario}_{flag}.csv",
            index=False,
        )
        print(
            f"ERROR: [{flag}] QEXTRA pass failed entirely (0/{len(survived_run_ids)} "
            "members) - all attempted members tracked as missing for this flag; "
            "skipping to the next flag."
        )
        continue

    n_qextra_survivors = (
        qextra_gsat_by_flag[flag].index.get_level_values("run_id").nunique()
    )
    if n_qextra_survivors < len(expected_run_ids):
        n_missing = len(expected_run_ids) - n_qextra_survivors
        pct = 100 * n_missing / len(expected_run_ids)
        print(
            f"WARNING: [{flag}] {n_missing}/{len(expected_run_ids)} ({pct:.1f}%) "
            f"member(s) missing after the QEXTRA pass - proceeding with "
            f"{n_qextra_survivors} survivors; gap will be tracked for backfill."
        )

In [ ]:
gsat_results_by_flag = {
    "ANTHROPOGENIC": anthropogenic_gsat,
    **qextra_gsat_by_flag,
}
gsat_results_by_flag

In [ ]:
{flag: r.index.get_level_values("run_id").nunique() for flag, r in gsat_results_by_flag.items()}

In [ ]:
# Create output directory for RAW (pre-calibration) results. Calibration - matching
# each ensemble's median to the AR6/AR7 assessed historical warming, via
# CMIP7ScenarioMIPPostProcessor - is applied once, separately, AFTER the full batch
# (cluster runs + any local backfill from 303_run_magicc-missing.ipynb) is complete
# for every scenario - see 304_post_process_gsat.ipynb. Applying it here, per-scenario,
# per-batch, would be wrong: the calibration shift it computes is a single scalar
# derived from the *median across whichever ensemble members happen to be present* in
# that specific call (see notes/ for the full trace through gcages' source) - so
# post-processing a partial/backfilled batch separately from the main batch bakes in a
# spurious, systematic offset between rows that should share one calibration.
output_dir = OUTPUT_FOLDER / "individual_runs_raw"
output_dir.mkdir(parents=True, exist_ok=True)

# Tracks any missing ensemble members per (model, scenario, flag) - see the harvest/
# QEXTRA cells above. 303_run_magicc-missing.ipynb reads these to drive a targeted
# local backfill re-run once the full batch is done, rather than losing members
# outright or re-running whole scenarios for the sake of a handful of members.
missing_members_dir = OUTPUT_FOLDER / "missing_members"
missing_members_dir.mkdir(parents=True, exist_ok=True)

print(f"Model: {MODEL}")
print(f"Scenario: {SCENARIO}")
print(f"Sanitized model: {sanitized_model}")
print(f"Sanitized scenario: {sanitized_scenario}")

# Save all three flags' raw results (manifest is handled by 302_running_helpers.py per session)
for flag, gsat_results in gsat_results_by_flag.items():
    output_filename = f"{sanitized_model}_{sanitized_scenario}_{flag}.csv"
    output_path = output_dir / output_filename
    gsat_results.to_csv(output_path)

    actual_run_ids = set(gsat_results.index.get_level_values("run_id"))
    missing_run_ids = sorted(set(expected_run_ids) - actual_run_ids)
    print(
        f"  {flag} raw output saved: {output_path} "
        f"({len(actual_run_ids)}/{len(expected_run_ids)} members)"
    )
    if missing_run_ids:
        missing_path = (
            missing_members_dir / f"{sanitized_model}_{sanitized_scenario}_{flag}.csv"
        )
        pd.DataFrame({
            "model": MODEL,
            "scenario": SCENARIO,
            "flag": flag,
            "run_id": missing_run_ids,
        }).to_csv(missing_path, index=False)
        print(f"    -> {len(missing_run_ids)} missing member(s) tracked: {missing_path}")